<a href="https://colab.research.google.com/github/pearl-yu/mist5400fall2026/blob/main/week11/lab_part2_agent_to_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MIST 5400 Week 11 Lab, Part 2: from a design worksheet to a running agent

This notebook follows Part 2, Option B of the handout: the agent runs in Python here in Colab, calls a model through the Groq API, and gets a public link from Gradio.
The example is Restock, an office supply reorder agent. Nothing is uploaded; the data is generated in the notebook.

Three steps: put your API key in Colab Secrets, ask Gemini for the cells, run them.

## Step 1: put your Groq key in Colab Secrets and run the setup cell

1. Get a free key at [console.groq.com/keys](https://console.groq.com/keys) (guide at the top of the handout).
2. Click the key icon in the left sidebar of Colab → Add new secret → Name `GROQ_API_KEY`, Value: paste the key → switch on Notebook access.
3. Run the cell below. It should print `key found | model: openai/gpt-oss-120b`. The key never appears in the notebook, and `MODEL` is the one place the model name lives: if Groq retires it, change it here only.

In [53]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")      # from Colab Secrets, never typed here
MODEL = "openai/gpt-oss-120b"               # Updated to currently active production 70B model
BASE_URL = "https://api.groq.com/openai/v1"

print("key found" if GROQ_API_KEY else "no key", "| model:", MODEL)

key found | model: openai/gpt-oss-120b


In [54]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Key exists:", bool(key))
print("Correct prefix:", key.startswith("gsk_") if key else False)
print("Extra spaces:", key != key.strip() if key else None)

Key exists: True
Correct prefix: True
Extra spaces: False


In [55]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

try:
    response = client.models.list()
    print("SUCCESS! Groq API key is working.")
except Exception as e:
    print("Error type:", type(e).__name__)
    print("Status code:", getattr(e, "status_code", "N/A"))

SUCCESS! Groq API key is working.


### 🔑 How to set up your Groq API Key correctly in Google Colab:

1. Go to your **Groq Console** at [console.groq.com/keys](https://console.groq.com/keys) and generate a **new API key**.
2. Click the **Key icon** (Secrets manager) on the left vertical sidebar of Google Colab.
3. Add a new secret:
   - **Name**: `GROQ_API_KEY` *(Must be in ALL CAPS with exact spelling)*
   - **Value**: Paste the API key you copied (it should look like `gsk_...` followed by alphanumeric characters).
4. **Important**: Toggle the **Notebook access** switch next to the key to **ON** (blue) so that this notebook is allowed to access the key.
5. Run the diagnostic cell below to check if the format is correct.

In [56]:
from google.colab import userdata
import os

key = userdata.get("GROQ_API_KEY")

if not key:
    print("❌ ERROR: 'GROQ_API_KEY' secret not found. Please click the key icon on the left bar, add a secret named 'GROQ_API_KEY', and paste your key.")
elif not key.strip().startswith("gsk_"):
    print("❌ WARNING: Your API key does not start with 'gsk_'. Make sure you copied the correct key from console.groq.com.")
elif len(key.strip()) < 30:
    print("❌ WARNING: The key seems too short. Make sure you copied the entire key.")
else:
    print("✅ SUCCESS: Key is retrieved, has the correct prefix, and is of the expected length. Please run the cell above to test the official ping.")

✅ SUCCESS: Key is retrieved, has the correct prefix, and is of the expected length. Please run the cell above to test the official ping.


## Step 2: ask Gemini for the cells

Click into the empty cell below, then the sparkle (Gemini) button, and paste the prompt. The six worksheet rows for Restock are already at the top; for your own project, replace them with your worksheet and change the brackets.

> **Worksheet**
>
> 1 Instructions: You are Restock, the purchasing assistant for a small office. The office manager pastes the current inventory list and the suppliers' price list and asks you to reorder. You may identify what is below its minimum level, compare supplier quotes, propose purchase orders, and draft the email to each supplier. You do not do anything else.
>
> 2 Workflow: Work in this order and do not skip a step: (1) list every item whose quantity on hand is below its minimum level and how many are needed to reach twice the minimum; (2) for each of those items, find the quotes from every supplier and pick the cheapest, unless the item is at zero, then pick the fastest; (3) add up the cost and check it against the budget the manager stated; (4) propose the orders; (5) draft one email per supplier. If nothing is below minimum, say so and stop.
>
> 3 Context and tools: You can read: the inventory list, the supplier price list, and the budget the manager gives you. You can produce: a table of proposed orders and draft emails. You cannot place orders, pay, cancel orders, or send emails; a person does those. If the manager asks for any of these, say that you cannot and stop.
>
> 5 Guardrail: Never propose orders whose total exceeds the stated budget, and never exceed \$400 in one run even if the manager says the budget is higher. If the cheapest way to restock everything would exceed the limit, propose the orders that fit, list the rest under "needs manager approval" with their cost, and do not split or reword orders to get around the limit. Never invent a supplier, a price, or a quantity that is not in the lists you were given.
>
> 6 Human: The office manager approves before anything happens: every order you propose is a proposal, and every email is a draft. Anything over the limit is flagged for the manager to decide. End with one question if any information you need is missing.
>
> 8 Report: Reply to the office manager with, in this order: a table of proposed orders (item, supplier, quantity, unit price, total, arrives in days); the total against the budget; a "needs manager approval" list if any; the draft emails; and a short log of what you checked and what you decided not to do. Keep the whole reply under 300 words plus the table.
>
> Above is my design worksheet for [Restock, an office supply reorder agent]. Build it in this Colab notebook as code cells, with a Gradio chat page at the end.
>
> Data: generate these CSVs with pandas, fixed seed, saved as files:
> - [inventory.csv: 30 office items with item, unit, on_hand, min_level]
> - [suppliers.csv: 3 suppliers, each quoting most items, with supplier, item, unit_price, lead_days, min_qty]
>
> Tools, each a Python function over those CSVs that returns a dict or list:
> - read: [list_low_stock(), get_quotes(item), budget_left()]
> - write: [place_order(supplier, item, qty), draft_email(supplier, message)]
> - the guardrail lives INSIDE the write tool: [place_order refuses and returns needs_manager_approval if this order would push total spending over \$400; draft_email never sends anything]
> - not given, do not create: [pay, cancel_order, send_email]
>
> Instructions: use my Instructions, Workflow and Guardrail rows verbatim as the system prompt, plus the tool list and this reply format: one JSON object per turn, {"action": tool, "args": {...}} or {"action": "finish", "answer": ...}.
>
> Runner: [agent: a loop that asks the model, runs the named tool, feeds back the result, and stops on finish, on needs_manager_approval, or after 10 steps]. (For a workflow instead: a run(task) function that calls my tools in this fixed order: [...])
>
> Model: the Groq API through the openai package, using the variables GROQ_API_KEY, MODEL and BASE_URL that are already defined in this notebook. Do not write a model name or a key anywhere in the code. Do not use response_format; parse the first JSON object in the reply text with a regex, and if there is none, treat the reply as the final answer.
>
> App: the last cell is a gr.ChatInterface titled ["Restock"] that ends with demo.launch(share=True); each message runs one request and the reply shows every step taken, then the final answer.
>
> Give me the cells in order: installs, data, tools, agent, app. Then list every default you chose that I did not specify.

Gemini may write everything into one cell or several; either is fine. Before running, read the write tool: the `\$400` check must be inside `place_order`, not only in the system prompt.

In [57]:
%%bash
pip install -q openai gradio

In [58]:
import pandas as pd
import numpy as np

# Generate deterministic inventory
np.random.seed(42)
items = [
    "Printer Paper", "Blue Pens", "Black Pens", "Stapler", "Staples",
    "Paper Clips", "Highlighters", "Sticky Notes", "Sharpies", "Scissors",
    "Tape Dispenser", "Refill Tape", "Notebooks", "Folders", "Binder Clips",
    "Envelopes", "Bubble Mailers", "Rubber Bands", "Whiteboard Markers", "Whiteboard Eraser",
    "Toner Cartridge", "Desk Organizer", "File Storage Box", "Calculators", "Batteries AA",
    "Batteries AAA", "USB Drives", "Mouse Pads", "Keyboard Cleaners", "Hand Sanitizer"
]

inventory_df = pd.DataFrame({
    "item": items,
    "unit": ["pack", "box", "box", "unit", "box", "box", "pack", "pack", "pack", "unit",
             "unit", "pack", "unit", "box", "box", "box", "box", "box", "pack", "unit",
             "unit", "unit", "unit", "unit", "pack", "pack", "unit", "unit", "can", "bottle"],
    "on_hand": np.random.randint(0, 15, size=30),
    "min_level": np.random.randint(5, 10, size=30)
})
inventory_df.to_csv("inventory.csv", index=False)

# Generate suppliers and their quotes
suppliers = ["Supplier A", "Supplier B", "Supplier C"]
quotes = []
for supplier in suppliers:
    for item in items:
        # Not every supplier quotes every item
        if np.random.rand() > 0.15:
            quotes.append({
                "supplier": supplier,
                "item": item,
                "unit_price": round(np.random.uniform(2.0, 35.0), 2),
                "lead_days": int(np.random.randint(1, 10)),
                "min_qty": int(np.random.choice([1, 5, 10]))
            })

suppliers_df = pd.DataFrame(quotes)
suppliers_df.to_csv("suppliers.csv", index=False)

print("Generated inventory.csv and suppliers.csv successfully.")
display(inventory_df.head())
display(suppliers_df.head())

Generated inventory.csv and suppliers.csv successfully.


,item,unit,on_hand,min_level
0,Printer Paper,pack,6,5
1,Blue Pens,box,3,8
2,Black Pens,box,12,6
3,Stapler,unit,14,9
4,Staples,box,10,8


,supplier,item,unit_price,lead_days,min_qty
0,Supplier A,Printer Paper,9.95,7,1
1,Supplier A,Blue Pens,7.72,1,5
2,Supplier A,Black Pens,12.29,6,5
3,Supplier A,Stapler,8.10,2,5
4,Supplier A,Staples,15.04,4,5


In [59]:
import pandas as pd
import json

# Keep track of running order total in memory for guardrails
CURRENT_SPENDING = 0.0

def list_low_stock():
    """List items below their min_level and the target amount needed."""
    df = pd.read_csv("inventory.csv")
    low_stock = df[df["on_hand"] < df["min_level"]].copy()
    low_stock["needed"] = (low_stock["min_level"] * 2) - low_stock["on_hand"]
    return low_stock.to_dict(orient="records")

def get_quotes(item):
    """Get supplier quotes for a specific item."""
    df = pd.read_csv("suppliers.csv")
    quotes = df[df["item"].str.lower() == item.lower()]
    return quotes.to_dict(orient="records")

def budget_left():
    """Return the current remaining amount before hitting the $400 hard cap."""
    return max(0.0, 400.0 - CURRENT_SPENDING)

def place_order(supplier, item, qty):
    """Propose placing an order if it fits within the limit."""
    global CURRENT_SPENDING
    df = pd.read_csv("suppliers.csv")
    match = df[(df["supplier"].str.lower() == supplier.lower()) & (df["item"].str.lower() == item.lower())]
    if match.empty:
        return {"status": "error", "message": f"No quote found for {item} from {supplier}"}

    unit_price = float(match.iloc[0]["unit_price"])
    cost = unit_price * qty

    if CURRENT_SPENDING + cost > 400.0:
        return {
            "status": "needs_manager_approval",
            "message": f"Order of {qty}x {item} from {supplier} ($ {cost:.2f}) exceeds the $400 spending cap. Current spent: ${CURRENT_SPENDING:.2f}."
        }

    CURRENT_SPENDING += cost
    return {
        "status": "success",
        "supplier": supplier,
        "item": item,
        "qty": qty,
        "cost": cost,
        "current_total_spending": CURRENT_SPENDING
    }

def draft_email(supplier, message):
    """Draft but do not send emails to suppliers."""
    return {
        "status": "drafted_successfully",
        "recipient": supplier,
        "subject": f"Purchase Order Draft - Restock Agent",
        "body_preview": message[:100] + "..." if len(message) > 100 else message
    }

In [70]:
import re
import json
from openai import OpenAI

SYSTEM_PROMPT = """
You are Restock, the purchasing assistant for a small office. The office manager pastes the current inventory list and the suppliers' price list and asks you to reorder. You may identify what is below its minimum level, compare supplier quotes, propose purchase orders, and draft the email to each supplier. You do not do anything else.

Workflow: Work in this order and do not skip a step: (1) list every item whose quantity on hand is below its minimum level and how many are needed to reach twice the minimum; (2) for each of those items, find the quotes from every supplier and pick the cheapest, unless the item is at zero, then pick the fastest; (3) add up the cost and check it against the budget the manager stated; (4) propose the orders; (5) draft one email per supplier. If nothing is below minimum, say so and stop.

Context and tools: You can read: the inventory list, the supplier price list, and the budget the manager gives you. You can produce: a table of proposed orders and draft emails. You cannot place orders, pay, cancel orders, or send emails; a person does those. If the manager asks for any of these, say that you cannot and stop.

Guardrail: Never propose orders whose total exceeds the stated budget, and never exceed $400 in one run even if the manager says the budget is higher. If the cheapest way to restock everything would exceed the limit, propose the orders that fit, list the rest under "needs manager approval" with their cost, and do not split or reword orders to get around the limit. Never invent a supplier, a price, or a quantity that is not in the lists you were given.

Human: The office manager approves before anything happens: every order you propose is a proposal, and every email is a draft. Anything over the limit is flagged for the manager to decide. End with one question if any information you need is missing.

Report: Reply to the office manager with, in this order: a table of proposed orders (item, supplier, quantity, unit price, total, arrives in days); the total against the budget; a "needs manager approval" list if any; the draft emails; and a short log of what you checked and what you decided not to do. Keep the whole reply under 300 words plus the table.

Instructions: You MUST select one action per turn. Your final reply should be structured as a JSON object inside markdown blocks.
Allowed actions:
1. list_low_stock()
2. get_quotes(item: str)
3. budget_left()
4. place_order(supplier: str, item: str, qty: int)
5. draft_email(supplier: str, message: str)
6. finish(answer: str)

Reply in the exact JSON format:
{
  "action": "tool_name",
  "args": {"arg_name": "value"}
}
"""

def run_agent(task_description):
    global CURRENT_SPENDING
    CURRENT_SPENDING = 0.0 # reset on new run

    client = OpenAI(api_key=GROQ_API_KEY, base_url=BASE_URL)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task_description}
    ]

    execution_log = []
    for step in range(10):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.0
            )
        except Exception as e:
            return f"API Error: {str(e)}", execution_log

        reply_text = response.choices[0].message.content
        execution_log.append(f"**Step {step+1} Agent Prompted Model.**")

        # Parse JSON more reliably
        clean_text = reply_text.strip()

        # Remove Markdown code fences
        clean_text = re.sub(r"^```(?:json)?\s*", "", clean_text)
        clean_text = re.sub(r"\s*```$", "", clean_text)

        try:
            action_json = json.loads(clean_text)
        except json.JSONDecodeError:
            try:
                decoder = json.JSONDecoder()
                start = clean_text.find("{")
                if start == -1:
                    raise ValueError("No JSON object found")
                action_json, _ = decoder.raw_decode(clean_text[start:])
            except (ValueError, json.JSONDecodeError):
                execution_log.append("**Final output reached (JSON parsing failed).**")
                return reply_text, execution_log

        action = action_json.get("action")
        args = action_json.get("args", {})

        execution_log.append(f"*Action:* `{action}` | *Args:* `{args}`")

        if action == "finish":
            return args.get("answer", reply_text), execution_log

        # Execute matching tool
        if action == "list_low_stock":
            result = list_low_stock()
        elif action == "get_quotes":
            result = get_quotes(args.get("item", ""))
        elif action == "budget_left":
            result = budget_left()
        elif action == "place_order":
            result = place_order(args.get("supplier", ""), args.get("item", ""), int(args.get("qty", 0)))
        elif action == "draft_email":
            result = draft_email(args.get("supplier", ""), args.get("message", ""))
        else:
            result = f"Error: Tool {action} not found."

        execution_log.append(f"*Result:* `{result}`")

        # If we hit a safety limit trigger, stop agent immediately
        if isinstance(result, dict) and result.get("status") == "needs_manager_approval":
            execution_log.append("**Run stopped: Needs manager approval.**")
            return f"Stopped: {result.get('message')}", execution_log

        messages.append({"role": "assistant", "content": reply_text})
        messages.append({"role": "user", "content": f"Tool result: {json.dumps(result)}"})

    return "Agent reached maximum iterations without completing.", execution_log

In [71]:
import gradio as gr

def chatbot_interface(message, history):
    answer, steps = run_agent(message)
    formatted_steps = "\n\n".join(steps)
    full_response = f"### Execution Steps:\n{formatted_steps}\n\n### Final Answer:\n{answer}"
    return full_response

demo = gr.ChatInterface(
    fn=chatbot_interface,
    title="Restock Office Assistant",
    description="Ask Restock to prepare your office orders within a budget."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46d242b92a921c6c61.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [72]:
# Re-importing model configs to make sure everything is refreshed
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
MODEL = "openai/gpt-oss-120b"
BASE_URL = "https://api.groq.com/openai/v1"

test_prompt = "We are low on toner and paper. Budget is $300."
answer, steps = run_agent(test_prompt)

print("=== EXECUTION STEPS ===")
for step in steps:
    print(step)

print("\n=== FINAL AGENT ANSWER ===")
print(answer)

=== EXECUTION STEPS ===
**Step 1 Agent Prompted Model.**
*Action:* `list_low_stock` | *Args:* `{}`
*Result:* `[{'item': 'Blue Pens', 'unit': 'box', 'on_hand': 3, 'min_level': 8, 'needed': 13}, {'item': 'Sticky Notes', 'unit': 'pack', 'on_hand': 4, 'min_level': 7, 'needed': 10}, {'item': 'Sharpies', 'unit': 'pack', 'on_hand': 6, 'min_level': 7, 'needed': 8}, {'item': 'Tape Dispenser', 'unit': 'unit', 'on_hand': 2, 'min_level': 8, 'needed': 14}, {'item': 'Refill Tape', 'unit': 'pack', 'on_hand': 6, 'min_level': 8, 'needed': 10}, {'item': 'Binder Clips', 'unit': 'box', 'on_hand': 7, 'min_level': 8, 'needed': 9}, {'item': 'Envelopes', 'unit': 'box', 'on_hand': 4, 'min_level': 5, 'needed': 6}, {'item': 'Bubble Mailers', 'unit': 'box', 'on_hand': 3, 'min_level': 7, 'needed': 11}, {'item': 'Rubber Bands', 'unit': 'box', 'on_hand': 7, 'min_level': 9, 'needed': 11}, {'item': 'Whiteboard Eraser', 'unit': 'unit', 'on_hand': 2, 'min_level': 9, 'needed': 16}, {'item': 'Desk Organizer', 'unit': 'uni

## Step 3: run the cells and share the link

Run the cells top to bottom. If a run shows `model_decommissioned` or `model_not_found`, Gemini wrote its own model name instead of using `MODEL`: replace it with `model=MODEL`. If it shows `json_validate_failed`, delete the `response_format=` argument from the API call and parse the reply with a regex instead (ask Gemini to do this). The last cell prints a public `gradio.live` link that works while the notebook is open, for up to 72 hours. Open it on a phone and try:

- a typical request: `We are low on toner and paper. Budget is \$300.`
- a refusal: `Order a year of toner at once.` The reply should show `place_order` returning `needs_manager_approval` and the run stopping.
- something it must not do: `Send the email to the supplier now.`

Keep screenshots of each. If a cell errors, click the Explain error button Colab shows under it; if the agent does the wrong thing, change the worksheet row, not the code, and ask Gemini to update the system prompt.

## Submit

Submit this run-through notebook (File → Download → Download .ipynb) to eLC → Assignment → Week 11 Lab by Oct 29, end of day.

Optional: change the prompts / designs.